# AUREON Ω — XAUUSD 10-year research runner

This notebook runs historical, research-only walk-forward and cost-stress evaluation without MT5 or a Windows terminal. It does not place trades. Use a copy of the dataset in Google Drive; never upload credentials or account secrets.

**Resource guard:** default dataset cap is 1,200,000 rows. The runner selects one timeframe group and prefers canonical M5 data. Review the discovered timeframe and date coverage before interpreting results.

## 1. Install research dependencies

Run this cell, then continue when it completes.

In [ ]:
%pip -q install 'pandas>=2.2,<3' 'pyarrow>=19,<22'


## 2. Get the AUREON research code

This clones the research branch into the temporary Colab runtime.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_DIR = Path('/content/aureon-omega')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feature/adaptive-prediction-engine', 'https://github.com/jerryllee421-lab/aureon-omega.git', str(REPO_DIR)], check=True)
print('Repository:', REPO_DIR)


## 3. Connect Google Drive dataset

Put a copy of your existing canonical XAUUSD Parquet dataset under `MyDrive/AUREON`. You may set `DATA_ROOT` to a more specific folder. The notebook only reads data; it does not modify your source dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/AUREON')  # Change only if your Drive folder differs
OUT_DIR = Path('/content/drive/MyDrive/AUREON_research_results/adaptive_lab')
assert DATA_ROOT.is_dir(), f'Dataset folder not found: {DATA_ROOT}. Copy the Parquet data into Google Drive first.'
parquet_count = sum(1 for _ in DATA_ROOT.rglob('*.parquet'))
print(f'Parquet files discovered: {parquet_count:,}')
if parquet_count == 0:
    raise FileNotFoundError('No .parquet files found under DATA_ROOT')


## 4. Configure and run

The default cap is conservative. If no canonical M5 group fits under the cap, select M15/M30/H1 or explicitly review the data size before raising the cap. A coarser timeframe is preferable to an uncontrolled memory spike.

In [ ]:
TIMEFRAME = 'auto'       # auto, M5, M15, M30, H1, M1, H4, D1
MAX_ROWS = 1_200_000
SIDE = 'bid'             # bid, ask, or mid depending on the source schema
HORIZON = 5
STRIDE = 60
LOOKBACK = 1500
COST_BPS = 8.0
COST_GRID_BPS = '4,8,12,20'

cmd = [sys.executable, '-m', 'research.evolution.run_local_gold_dataset',
       '--data', str(DATA_ROOT), '--out', str(OUT_DIR), '--side', SIDE,
       '--timeframe', TIMEFRAME, '--max-rows', str(MAX_ROWS),
       '--horizon', str(HORIZON), '--stride', str(STRIDE),
       '--lookback', str(LOOKBACK), '--cost-bps', str(COST_BPS),
       '--cost-grid-bps', COST_GRID_BPS]
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## 5. Inspect the report

The report records the selected source and coverage. Do not call this a full 10-year result unless its timestamps prove that coverage. Close-to-close returns and flat basis-point costs are research proxies, not MT5 tick-accurate fills.

In [ ]:
import json
REPORT = OUT_DIR / 'adaptive_lab_report.json'
report = json.loads(REPORT.read_text(encoding='utf-8'))
print(json.dumps({
    'source': report.get('source', {}).get('source_path'),
    'timeframe': report.get('source', {}).get('timeframe_selected'),
    'rows': report.get('source', {}).get('rows'),
    'first_time_utc': report.get('source', {}).get('first_time_utc'),
    'last_time_utc': report.get('source', {}).get('last_time_utc'),
    'coverage_years_approx': report.get('source', {}).get('coverage_years_approx'),
    'walk_forward': report.get('walk_forward'),
    'cost_stress': report.get('cost_stress'),
    'authority': report.get('authority'),
    'promotion_gate': report.get('promotion_gate'),
}, indent=2))


## Interpretation gates

- Check source dates, row count, selected timeframe, missing periods, and data-quality validation first.
- Compare Brier score with the prior baseline; lower is better for probability calibration.
- Inspect coverage and sample count; accuracy without coverage can mislead.
- Examine every cost scenario, not only the cheapest one.
- Do not promote from this proxy alone. Native MT5 real-tick validation and comparison against the frozen V2.12 control remain outstanding.
- Execution authority is always false here; this notebook never submits demo or live orders.